# Circuit-Level Posterior-Symmetric Half-Null Audit

This notebook investigates complementary half-null pairs at the projected circuit-decoder level.

For a row in `ker([Hz; Lz])`, complementary halves should have the same syndrome and the same projected logical effect. The missing ingredient is posterior symmetry: if the two halves do not have the same prior-weighted posterior mass, BP may still prefer one representative even though the detector input is identical.

This notebook therefore separates three questions:
- do complementary halves give the same projected decoder input?
- do they have the same posterior under `error_priors`?
- when BP picks one representative, does it match half A, half B, or neither?

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from relay_bp.analysis import (
    audit_half_null_posteriors,
    compute_half_null_pair_posteriors,
    default_export_code_paths,
    enumerate_half_null_cases,
    load_code_capacity_problem,
    pair_half_null_cases,
    summarize_half_null_pair_posteriors,
)

smoke_problem = load_code_capacity_problem(default_export_code_paths(REPO_ROOT)["surface13"])
smoke_cases = enumerate_half_null_cases(smoke_problem)
smoke_pairs = pair_half_null_cases(smoke_cases)
smoke_pair_rows = compute_half_null_pair_posteriors(smoke_problem, smoke_pairs)
smoke_pair_class_counts = summarize_half_null_pair_posteriors(smoke_pair_rows)
smoke_audit = audit_half_null_posteriors(
    problem=smoke_problem,
    pairs=smoke_pairs,
    max_iter=20,
    alpha=1.0,
    max_pairs_per_class=1,
)
assert smoke_pair_class_counts["exact_equal"]["num_pairs"] >= 1
assert smoke_audit["summary"]["decoder_output_mismatches"] == 0
assert smoke_audit["summary"]["logical_success_mismatches"] == 0
smoke_pair_class_counts["exact_equal"], smoke_audit["decoded_matches_counts"]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

code_names = ["gross", "two_gross", "surface13"]
posterior_rows_by_code = {}
audits_by_code = {}
pairs_by_code = {}
problems_by_code = {}
overview_rows = []

for code_name in code_names:
    problem = load_code_capacity_problem(default_export_code_paths(REPO_ROOT)[code_name])
    problems_by_code[code_name] = problem
    cases = enumerate_half_null_cases(problem)
    pairs = pair_half_null_cases(cases)
    pairs_by_code[code_name] = pairs
    posterior_rows = compute_half_null_pair_posteriors(problem, pairs)
    posterior_rows_by_code[code_name] = pd.DataFrame(posterior_rows)
    audits_by_code[code_name] = audit_half_null_posteriors(
        problem=problem,
        pairs=pairs,
        max_iter=40,
        alpha=1.0,
    )
    overview_rows.append(
        {
            "code": code_name,
            **audits_by_code[code_name]["summary"],
            **{
                f"{pair_class}_pairs": audits_by_code[code_name]["pair_class_counts"][pair_class]["num_pairs"]
                for pair_class in ["exact_equal", "near_equal", "mild_bias", "biased", "strong_bias"]
            },
        }
    )

overview_df = pd.DataFrame(overview_rows)
overview_df


In [ ]:
fig, axes = plt.subplots(1, len(code_names), figsize=(15.0, 4.0), constrained_layout=True)
for ax, code_name in zip(axes, code_names):
    gap_values = posterior_rows_by_code[code_name]["posterior_gap_log_odds"].to_numpy(dtype=float)
    ax.hist(np.log10(np.maximum(gap_values, 1e-15)), bins=24, color="#3f6db3", alpha=0.85)
    ax.set_title(code_name)
    ax.set_xlabel("log10 posterior gap")
    ax.set_ylabel("pair count")
plt.show()


In [ ]:
selected_code = "surface13"
selected_audit = audits_by_code[selected_code]
selected_class_summary_df = pd.DataFrame(
    [
        {"pair_class": pair_class, **selected_audit["pair_class_counts"][pair_class], **selected_audit["pair_class_summary"][pair_class]}
        for pair_class in ["exact_equal", "near_equal", "mild_bias", "biased", "strong_bias"]
    ]
)
selected_exact_equal_manifest_df = pd.DataFrame(selected_audit["representative_examples"]["exact_equal"])
selected_class_summary_df, selected_exact_equal_manifest_df[[
    "pair_id",
    "posterior_gap_log_odds",
    "half_a_bits",
    "half_b_bits",
    "decoded_bits",
    "decoded_matches",
    "decoder_output_equal",
    "exact_recovery_split",
]]


In [ ]:
from relay_bp.analysis import trace_half_null_pair


def select_pair_for_class(code_name: str, pair_class: str):
    examples = audits_by_code[code_name]["representative_examples"][pair_class]
    if not examples:
        return None
    pair_id = int(examples[0]["pair_id"])
    return next(pair for pair in pairs_by_code[code_name] if int(pair["pair_id"]) == pair_id)


trace_pair_classes = ["exact_equal", "near_equal", "strong_bias"]
trace_pairs = {pair_class: select_pair_for_class(selected_code, pair_class) for pair_class in trace_pair_classes}
trace_artifacts = {
    pair_class: (
        None
        if pair is None
        else trace_half_null_pair(
            problem=problems_by_code[selected_code],
            pair=pair,
            max_iter=40,
            alpha=1.0,
        )
    )
    for pair_class, pair in trace_pairs.items()
}
trace_selection_df = pd.DataFrame(
    [
        {
            "pair_class": pair_class,
            "pair_id": None if trace_pairs[pair_class] is None else int(trace_pairs[pair_class]["pair_id"]),
            "available": trace_pairs[pair_class] is not None,
            "same_posterior_trace": None if trace_artifacts[pair_class] is None else bool(trace_artifacts[pair_class]["same_posterior_trace"]),
            "same_decoding": None if trace_artifacts[pair_class] is None else bool(trace_artifacts[pair_class]["same_decoding"]),
        }
        for pair_class in trace_pair_classes
    ]
)
trace_selection_df


In [ ]:
fig, axes = plt.subplots(1, len(trace_pair_classes), figsize=(16.0, 4.5), constrained_layout=True)
for ax, pair_class in zip(axes, trace_pair_classes):
    pair = trace_pairs[pair_class]
    trace_artifact = trace_artifacts[pair_class]
    if pair is None or trace_artifact is None:
        ax.text(0.5, 0.5, f"No {pair_class} pair found\nunder current thresholds", ha="center", va="center")
        ax.set_axis_off()
        continue
    bit_to_plot = int(pair["support_bits"][0])
    ax.plot(trace_artifact["trace_a"]["iterations"], trace_artifact["trace_a"]["posterior_trace"][:, bit_to_plot], marker="o", label="half A")
    ax.plot(trace_artifact["trace_b"]["iterations"], trace_artifact["trace_b"]["posterior_trace"][:, bit_to_plot], marker="x", linestyle="--", label="half B")
    ax.set_title(f"{pair_class} trace")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(f"Posterior ratio on bit {bit_to_plot}")
    ax.legend()
plt.show()
